In [1]:
%load_ext autoreload
%autoreload 2
from notebook_cache import NotebookCache

cache = NotebookCache('../cache/uwm2')

# UWM 2 — does action-free video help when labels are scarce?

UWM's pitch: the frame half trains on video **without** actions, so demos nobody labelled still teach the shared body. Test on T1 with 100 demos (`pick_place_cube_n100`): 10 held out for val, 10 labelled, 80 video-only. Compare 10 labelled alone vs 10 labelled + 80 video-only. Model: `UWMxx` from uwm.ipynb (both heads predict clean), its pieces in `uwm_parts.py`.

### step 0 — the 100 demos, 10 held out


In [2]:
import numpy as np
import torch
from uwm_parts import load_table, load_top

NAME = 'pick_place_cube_n100'

In [3]:
@cache
def table():
    return load_table(NAME)  # state [N, 8], action [N, 8], episode [N]

table: cached, 1 call(s) stored


In [4]:
@cache
def images():
    return load_top(NAME)  # [N, 3, 128, 128] uint8, ~1 GB, a few minutes the first time

images: cached, 1 call(s) stored


In [5]:
state, action, episode = table()
top = images()
val_eps = np.random.default_rng(0).permutation(100)[:10]
is_val = np.isin(episode, val_eps)  # [N]
print(f'{state.shape=}  {top.shape=}  {val_eps=}')
print(f'train rows {(~is_val).sum()} (90 demos)   val rows {is_val.sum()} (10 demos)')

state.shape=(21295, 8)  top.shape=(21295, 3, 128, 128)  val_eps=array([82, 36, 20,  5, 93, 16, 94, 52, 72, 90])
train rows 19156 (90 demos)   val rows 2139 (10 demos)


### step 1 — the floor: `UWMxx` trained on the 10 labelled demos alone

10 of the 90 train demos keep their actions; the action and state stats come from those 10 only, as if the other 80 had never been labelled. Same 10k steps, batch 64, lr 1e-3 as uwm.ipynb (`UWMxx` there, 20 demos all labelled, from the 25-demo set: 32/40).


In [6]:
from uwm_parts import device, to_gpu

train_eps = np.setdiff1d(np.arange(100), val_eps)  # [90]
labelled = np.random.default_rng(1).choice(train_eps, 10, replace=False)
is_lab = np.isin(episode, labelled)  # [N]
d = to_gpu(state, action, episode, top, train=is_lab)
LAB = torch.tensor(np.flatnonzero(is_lab), device=device)  # [Nlab] rows with actions
VAL = torch.tensor(np.flatnonzero(is_val), device=device)
print(f'{labelled=}   labelled rows {len(LAB)}   val rows {len(VAL)}')

labelled=array([31, 42, 77,  2, 91, 45, 13, 67, 86, 25])   labelled rows 2116   val rows 2139


In [7]:
from uwm_parts import UWMxx, batch_loss_xx, val_loss_xx


@cache
def train_lab(seed: int = 0, steps: int = 10000, lr: float = 1e-3) -> dict[str, torch.Tensor]:
    # uwm.ipynb's train_xx, batches drawn from the labelled rows only
    torch.manual_seed(seed)
    net = UWMxx(d).to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=lr)
    for step in range(steps + 1):
        rows = LAB[torch.randint(0, len(LAB), (64,), device=device)]
        with torch.autocast('cuda', dtype=torch.bfloat16):
            loss_f, loss_a = batch_loss_xx(net, d, rows)
        opt.zero_grad()
        (loss_f + loss_a).backward()
        opt.step()
        if step % 500 == 0:
            val_f, val_a = val_loss_xx(net, d, VAL)
            print(
                f'{step:5d}  train frame {loss_f.item():.4f} chunk {loss_a.item():.4f}'
                f'   val frame {val_f:.4f} chunk {val_a:.4f}'
            )
    return {k: v.cpu() for k, v in net.state_dict().items()}

train_lab: cached, 1 call(s) stored


In [21]:
lab = UWMxx(d).to(device)
lab.load_state_dict(train_lab())
lab.eval();

    0  train frame 0.6589 chunk 0.9821   val frame 0.4855 chunk 1.1497
  500  train frame 0.0250 chunk 0.0164   val frame 0.0233 chunk 0.0425
 1000  train frame 0.0161 chunk 0.0097   val frame 0.0183 chunk 0.0310
 1500  train frame 0.0132 chunk 0.0064   val frame 0.0157 chunk 0.0314
 2000  train frame 0.0126 chunk 0.0038   val frame 0.0140 chunk 0.0284
 2500  train frame 0.0137 chunk 0.0058   val frame 0.0129 chunk 0.0317
 3000  train frame 0.0097 chunk 0.0055   val frame 0.0121 chunk 0.0304
 3500  train frame 0.0099 chunk 0.0096   val frame 0.0115 chunk 0.0321
 4000  train frame 0.0112 chunk 0.0044   val frame 0.0110 chunk 0.0312
 4500  train frame 0.0083 chunk 0.0048   val frame 0.0105 chunk 0.0336
 5000  train frame 0.0076 chunk 0.0024   val frame 0.0103 chunk 0.0316
 5500  train frame 0.0082 chunk 0.0040   val frame 0.0102 chunk 0.0334
 6000  train frame 0.0069 chunk 0.0022   val frame 0.0101 chunk 0.0360
 6500  train frame 0.0071 chunk 0.0022   val frame 0.0099 chunk 0.0359
 7000 

In [8]:
from uwm_parts import make_sim

env = make_sim()  # the T1 sim, as uwm.ipynb step 16

In [9]:
import contextlib
import io
from collections import Counter

from uwm_parts import rollout


@cache
def sweep_lab(n_exec: int, train_seed: int = 0, train_steps: int = 10000) -> dict[int, tuple]:
    # uwm.ipynb's sweep_xx, with the weights train_lab gives; out: {env seed: (ok, mode, steps)}
    net = UWMxx(d).to(device)
    with contextlib.redirect_stdout(io.StringIO()):  # train_lab's replayed log isn't this cell's
        net.load_state_dict(train_lab(seed=train_seed, steps=train_steps))
    net.eval()
    torch.manual_seed(n_exec)  # the sampling noise, fixed per call
    res = {seed: rollout(env, net, d, seed, n_exec) for seed in range(1000, 1020)}
    fails = Counter(mode for ok, mode, _ in res.values() if not ok)
    print(
        f'n_exec {n_exec:2d}: {sum(ok for ok, _, _ in res.values()):2d}/20   failures {dict(fails)}'
    )
    return res

sweep_lab: cached, 2 call(s) stored


In [16]:
sweeps_lab = {n_exec: sweep_lab(n_exec) for n_exec in [8, 16]}
print('UWMxx, 20 demos all labelled (uwm.ipynb): 16, 16')

n_exec  8: 16/20   failures {'no_grasp': 4}
n_exec 16: 14/20   failures {'dropped': 1, 'no_grasp': 5}
UWMxx, 20 demos all labelled (uwm.ipynb): 16, 16


### step 2 — a lower floor: 3 labelled demos alone

10 labelled alone already did 30/40 (20 labelled: 32/40), too close to the top for video to show. Take the first 3 of the 10 (the sets nest), stats from those 3 only, same training and sweep.


In [10]:
import dataclasses


def data_for(n_lab: int) -> tuple:
    # out: d with action/state stats from labelled[:n_lab] (frames shared, not copied), its rows
    mask = np.isin(episode, labelled[:n_lab])
    g = lambda x: torch.tensor(x, device=device)
    stats = dict(A_MEAN=action[mask].mean(0), A_STD=action[mask].std(0),
                 S_MEAN=state[mask].mean(0), S_STD=state[mask].std(0))  # fmt: skip
    return dataclasses.replace(d, **{k: g(v) for k, v in stats.items()}), g(np.flatnonzero(mask))


d3, LAB3 = data_for(3)
print(f'demos {labelled[:3]}   labelled rows {len(LAB3)}')

demos [31 42 77]   labelled rows 630


In [11]:
@cache
def train_few(n_lab: int, seed: int = 0, steps: int = 10000, lr: float = 1e-3) -> dict:
    # train_lab on labelled[:n_lab]
    dn, rows_lab = data_for(n_lab)
    torch.manual_seed(seed)
    net = UWMxx(dn).to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=lr)
    for step in range(steps + 1):
        rows = rows_lab[torch.randint(0, len(rows_lab), (64,), device=device)]
        with torch.autocast('cuda', dtype=torch.bfloat16):
            loss_f, loss_a = batch_loss_xx(net, dn, rows)
        opt.zero_grad()
        (loss_f + loss_a).backward()
        opt.step()
        if step % 500 == 0:
            val_f, val_a = val_loss_xx(net, dn, VAL)
            print(
                f'{step:5d}  train frame {loss_f.item():.4f} chunk {loss_a.item():.4f}'
                f'   val frame {val_f:.4f} chunk {val_a:.4f}'
            )
    return {k: v.cpu() for k, v in net.state_dict().items()}

train_few: cached, 1 call(s) stored


In [24]:
few3 = UWMxx(d3).to(device)
few3.load_state_dict(train_few(3))
few3.eval();

    0  train frame 0.6581 chunk 0.9688   val frame 0.4854 chunk 1.1046
  500  train frame 0.0221 chunk 0.0146   val frame 0.0238 chunk 0.1504
 1000  train frame 0.0148 chunk 0.0087   val frame 0.0186 chunk 0.1423
 1500  train frame 0.0124 chunk 0.0049   val frame 0.0165 chunk 0.1412
 2000  train frame 0.0096 chunk 0.0100   val frame 0.0155 chunk 0.1463
 2500  train frame 0.0110 chunk 0.0034   val frame 0.0148 chunk 0.1458
 3000  train frame 0.0084 chunk 0.0045   val frame 0.0144 chunk 0.1457
 3500  train frame 0.0076 chunk 0.0039   val frame 0.0148 chunk 0.1477
 4000  train frame 0.0076 chunk 0.0037   val frame 0.0149 chunk 0.1494
 4500  train frame 0.0067 chunk 0.0019   val frame 0.0154 chunk 0.1480
 5000  train frame 0.0062 chunk 0.0023   val frame 0.0161 chunk 0.1496
 5500  train frame 0.0059 chunk 0.0015   val frame 0.0166 chunk 0.1502
 6000  train frame 0.0053 chunk 0.0014   val frame 0.0173 chunk 0.1515
 6500  train frame 0.0054 chunk 0.0018   val frame 0.0179 chunk 0.1557
 7000 

In [20]:
@cache
def sweep_few(n_exec: int, n_lab: int, train_seed: int = 0, train_steps: int = 10000) -> dict:
    # sweep_lab with the weights train_few(n_lab) gives
    dn, _ = data_for(n_lab)
    net = UWMxx(dn).to(device)
    with contextlib.redirect_stdout(io.StringIO()):
        net.load_state_dict(train_few(n_lab, seed=train_seed, steps=train_steps))
    net.eval()
    torch.manual_seed(n_exec)  # the sampling noise, fixed per call
    res = {seed: rollout(env, net, dn, seed, n_exec) for seed in range(1000, 1020)}
    fails = Counter(mode for ok, mode, _ in res.values() if not ok)
    print(
        f'n_exec {n_exec:2d}: {sum(ok for ok, _, _ in res.values()):2d}/20   failures {dict(fails)}'
    )
    return res

sweep_few: not cached, will compute


In [21]:
sweeps_few3 = {n_exec: sweep_few(n_exec, 3) for n_exec in [8, 16]}
print('10 labelled: 16, 14;  20 labelled (uwm.ipynb): 16, 16')

n_exec  8:  6/20   failures {'no_grasp': 12, 'dropped': 2}
n_exec 16:  6/20   failures {'no_grasp': 13, 'wrong_target': 1}
10 labelled: 16, 14;  20 labelled (uwm.ipynb): 16, 16


### step 3 — 3 labelled + 87 video-only: does the unlabelled video lift the 12/40?

A video-only row has frames and joints but no actions: its chunk stays pure noise (τ_a = 0, UWM's "unknown") and gets no chunk loss, so only the frame half learns from it. Each step: 64 labelled rows + 64 video-only rows in one batch of 128; frame loss over all 128, chunk loss over the 64 labelled, so the chunk half gets exactly step 2's updates.


In [12]:
from uwm_parts import UWM, cut


def batch_loss_mix(net: UWMxx, dn, rows: torch.Tensor, has_act: torch.Tensor) -> tuple:
    # batch_loss_xx; rows with has_act False: chunk at pure noise (tau_a 0), no chunk loss
    # (their ACTION is read, but tau_a 0 multiplies it by zero: nothing leaks)
    clean_f = cut(dn.TOP[dn.AFTER[rows, 0]].float() / 255 * 2 - 1)  # [B, 64, 768]
    clean_a = (dn.ACTION[dn.CHUNK[rows]] - dn.A_MEAN) / dn.A_STD  # [B, 16, 8]
    noise_f, noise_a = torch.randn_like(clean_f), torch.randn_like(clean_a)
    tau_v, tau_a = torch.rand(2, len(rows), device=device)
    tau_a = tau_a * has_act  # [B], 0 on video-only rows
    x_f = (1 - tau_v[:, None, None]) * noise_f + tau_v[:, None, None] * clean_f
    x_a = (1 - tau_a[:, None, None]) * noise_a + tau_a[:, None, None] * clean_a
    seen = dn.TOP[dn.PAST[rows]].float() / 255
    f_hat, a_hat = UWM.forward(net, seen, dn.STATE[rows], x_f, x_a, tau_v, tau_a)
    keep = (dn.REAL[rows] & has_act[:, None])[..., None].float()  # [B, 16, 1]
    loss_a = (((a_hat - clean_a) ** 2) * keep).sum() / (keep.sum() * 8)
    return torch.nn.functional.mse_loss(f_hat, clean_f), loss_a


VID3 = torch.tensor(np.flatnonzero(~is_val & ~np.isin(episode, labelled[:3])), device=device)
has_act = torch.arange(128, device=device) < 64  # first 64 labelled, last 64 video-only
torch.manual_seed(1)
rows = torch.cat([LAB3[torch.randint(0, len(LAB3), (64,), device=device)],
                  VID3[torch.randint(0, len(VID3), (64,), device=device)]])  # fmt: skip
print(f'video-only rows {len(VID3)} (87 demos)')
print(
    'untrained frame {:.3f}  chunk {:.3f}'.format(
        *batch_loss_mix(UWMxx(d3).to(device), d3, rows, has_act)
    )
)

video-only rows 18526 (87 demos)
untrained frame 0.654  chunk 0.981


In [13]:
@cache
def train_mix(n_lab: int, seed: int = 0, steps: int = 10000, lr: float = 1e-3) -> dict:
    # train_few, each batch 64 labelled + 64 video-only rows (all other train demos)
    dn, rows_lab = data_for(n_lab)
    rows_vid = torch.tensor(
        np.flatnonzero(~is_val & ~np.isin(episode, labelled[:n_lab])), device=device
    )
    has_act = torch.arange(128, device=device) < 64
    torch.manual_seed(seed)
    net = UWMxx(dn).to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=lr)
    for step in range(steps + 1):
        rows = torch.cat([rows_lab[torch.randint(0, len(rows_lab), (64,), device=device)],
                          rows_vid[torch.randint(0, len(rows_vid), (64,), device=device)]])  # fmt: skip
        with torch.autocast('cuda', dtype=torch.bfloat16):
            loss_f, loss_a = batch_loss_mix(net, dn, rows, has_act)
        opt.zero_grad()
        (loss_f + loss_a).backward()
        opt.step()
        if step % 500 == 0:
            val_f, val_a = val_loss_xx(net, dn, VAL)
            print(
                f'{step:5d}  train frame {loss_f.item():.4f} chunk {loss_a.item():.4f}'
                f'   val frame {val_f:.4f} chunk {val_a:.4f}'
            )
    return {k: v.cpu() for k, v in net.state_dict().items()}

train_mix: cached, 1 call(s) stored


In [14]:
mix3 = UWMxx(d3).to(device)
mix3.load_state_dict(train_mix(3))
mix3.eval();

    0  train frame 0.6596 chunk 0.9688   val frame 0.4854 chunk 1.1052
  500  train frame 0.0216 chunk 0.0110   val frame 0.0228 chunk 0.1525
 1000  train frame 0.0192 chunk 0.0088   val frame 0.0179 chunk 0.1491
 1500  train frame 0.0157 chunk 0.0050   val frame 0.0152 chunk 0.1480
 2000  train frame 0.0141 chunk 0.0067   val frame 0.0137 chunk 0.1542
 2500  train frame 0.0118 chunk 0.0035   val frame 0.0124 chunk 0.1522
 3000  train frame 0.0115 chunk 0.0036   val frame 0.0116 chunk 0.1469
 3500  train frame 0.0099 chunk 0.0032   val frame 0.0111 chunk 0.1516
 4000  train frame 0.0101 chunk 0.0028   val frame 0.0104 chunk 0.1486
 4500  train frame 0.0090 chunk 0.0045   val frame 0.0097 chunk 0.1496
 5000  train frame 0.0088 chunk 0.0021   val frame 0.0091 chunk 0.1483
 5500  train frame 0.0087 chunk 0.0037   val frame 0.0088 chunk 0.1495
 6000  train frame 0.0077 chunk 0.0012   val frame 0.0086 chunk 0.1482
 6500  train frame 0.0070 chunk 0.0014   val frame 0.0083 chunk 0.1499
 7000 

In [25]:
@cache
def sweep_mix(n_exec: int, n_lab: int, train_seed: int = 0, train_steps: int = 10000) -> dict:
    # sweep_few with the weights train_mix(n_lab) gives
    dn, _ = data_for(n_lab)
    net = UWMxx(dn).to(device)
    with contextlib.redirect_stdout(io.StringIO()):
        net.load_state_dict(train_mix(n_lab, seed=train_seed, steps=train_steps))
    net.eval()
    torch.manual_seed(n_exec)  # the sampling noise, fixed per call
    res = {seed: rollout(env, net, dn, seed, n_exec) for seed in range(1000, 1020)}
    fails = Counter(mode for ok, mode, _ in res.values() if not ok)
    print(
        f'n_exec {n_exec:2d}: {sum(ok for ok, _, _ in res.values()):2d}/20   failures {dict(fails)}'
    )
    return res

sweep_mix: not cached, will compute


In [26]:
sweeps_mix3 = {n_exec: sweep_mix(n_exec, 3) for n_exec in [8, 16]}
print('3 labelled alone: 6, 6;  10 labelled: 16, 14')

n_exec  8:  8/20   failures {'no_grasp': 12}
n_exec 16:  7/20   failures {'no_grasp': 12, 'dropped': 1}
3 labelled alone: 6, 6;  10 labelled: 16, 14


### step 4 — can the chunk read a frame at all? Policy job vs inverse job on val

Video fixed the frame half (val frame 0.0218 → 0.0066) but not the actions. Before trying to make the actions use the imagined frame, check they can use a frame: give the **true** t+16 frame (τ_v = 1, uwm.ipynb step 15's inverse job) and compare first-action error with the policy job (frame left as noise). No training.


In [13]:
@torch.no_grad()
def first_error(net: UWMxx, dn, inverse: bool, steps: int = 10) -> torch.Tensor:
    # out: [8] val first-action |error|, radians; inverse: the true t+16 frame given at tau_v 1
    torch.manual_seed(0)
    errs = []
    for j in range(0, len(VAL), 256):
        rows = VAL[j : j + 256]
        B = len(rows)
        if inverse:
            x_f = cut(dn.TOP[dn.AFTER[rows, 0]].float() / 255 * 2 - 1)  # where it must end up
        else:
            x_f = torch.randn(B, 64, 768, device=device)  # left as noise
        tau_v = torch.full((B,), float(inverse), device=device)
        x_a = torch.randn(B, 16, 8, device=device)
        seen = dn.TOP[dn.PAST[rows]].float() / 255
        for k in range(steps):
            tau_a = torch.full((B,), k / steps, device=device)
            with torch.autocast('cuda', dtype=torch.bfloat16):
                _, v_a = net(seen, dn.STATE[rows], x_f, x_a, tau_v, tau_a)
            x_a = x_a + v_a.float() / steps
        errs.append(((x_a * dn.A_STD + dn.A_MEAN)[:, 0] - dn.ACTION[rows]).abs())
    return torch.cat(errs).mean(0)

In [16]:
models = {'3 alone': (few3, d3), '3 + video': (mix3, d3), '10 alone': (lab, d)}
print('                 policy job         inverse job')
for name, (net, dn) in models.items():
    p, i = first_error(net, dn, False), first_error(net, dn, True)
    print(
        f'{name:10s}  arm {p[:7].mean():.4f} grip {p[7]:.4f}   arm {i[:7].mean():.4f} grip {i[7]:.4f}'
    )

                 policy job         inverse job
3 alone     arm 0.0273 grip 0.0889   arm 0.0271 grip 0.0899
3 + video   arm 0.0260 grip 0.0372   arm 0.0259 grip 0.0372
10 alone    arm 0.0114 grip 0.0353   arm 0.0113 grip 0.0359


### step 5 — teach the chunk to read a frame: half the labelled rows get the true t+16 at τ_v = 1

Uniform τ_v almost never puts a near-clean frame next to the chunk, so it learned to ignore it. Each batch now: 32 labelled rows as before, 32 labelled rows with the true t+16 frame given (the inverse job), 64 video-only rows. The given-frame rows are left out of the frame loss (copying a given frame teaches nothing).


In [12]:
def batch_loss_inv(
    net: UWMxx, dn, rows: torch.Tensor, has_act: torch.Tensor, inv: torch.Tensor
) -> tuple:
    # batch_loss_mix; rows with inv True: frame given clean (tau_v 1), no frame loss
    clean_f = cut(dn.TOP[dn.AFTER[rows, 0]].float() / 255 * 2 - 1)  # [B, 64, 768]
    clean_a = (dn.ACTION[dn.CHUNK[rows]] - dn.A_MEAN) / dn.A_STD  # [B, 16, 8]
    noise_f, noise_a = torch.randn_like(clean_f), torch.randn_like(clean_a)
    tau_v, tau_a = torch.rand(2, len(rows), device=device)
    tau_v = torch.where(inv, 1.0, tau_v)  # [B], 1 on the inverse-job rows
    tau_a = tau_a * has_act  # [B], 0 on video-only rows
    x_f = (1 - tau_v[:, None, None]) * noise_f + tau_v[:, None, None] * clean_f
    x_a = (1 - tau_a[:, None, None]) * noise_a + tau_a[:, None, None] * clean_a
    seen = dn.TOP[dn.PAST[rows]].float() / 255
    f_hat, a_hat = UWM.forward(net, seen, dn.STATE[rows], x_f, x_a, tau_v, tau_a)
    keep = (dn.REAL[rows] & has_act[:, None])[..., None].float()  # [B, 16, 1]
    loss_a = (((a_hat - clean_a) ** 2) * keep).sum() / (keep.sum() * 8)
    loss_f = ((f_hat - clean_f) ** 2).mean((1, 2))[~inv].mean()  # per row, given-frame rows out
    return loss_f, loss_a


inv = (torch.arange(128, device=device) >= 32) & (
    torch.arange(128, device=device) < 64
)  # rows 32..63
torch.manual_seed(1)
print(
    'untrained frame {:.3f}  chunk {:.3f}'.format(
        *batch_loss_inv(UWMxx(d3).to(device), d3, rows, has_act, inv)
    )
)

untrained frame 0.654  chunk 0.981


In [18]:
@cache
def train_inv(n_lab: int, seed: int = 0, steps: int = 10000, lr: float = 1e-3) -> dict:
    # train_mix with batch_loss_inv: rows 0..31 labelled, 32..63 labelled + frame given, 64..127 video-only
    dn, rows_lab = data_for(n_lab)
    rows_vid = torch.tensor(
        np.flatnonzero(~is_val & ~np.isin(episode, labelled[:n_lab])), device=device
    )
    i = torch.arange(128, device=device)
    has_act, inv = i < 64, (i >= 32) & (i < 64)
    torch.manual_seed(seed)
    net = UWMxx(dn).to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=lr)
    for step in range(steps + 1):
        rows = torch.cat([rows_lab[torch.randint(0, len(rows_lab), (64,), device=device)],
                          rows_vid[torch.randint(0, len(rows_vid), (64,), device=device)]])  # fmt: skip
        with torch.autocast('cuda', dtype=torch.bfloat16):
            loss_f, loss_a = batch_loss_inv(net, dn, rows, has_act, inv)
        opt.zero_grad()
        (loss_f + loss_a).backward()
        opt.step()
        if step % 500 == 0:
            val_f, val_a = val_loss_xx(net, dn, VAL)
            print(
                f'{step:5d}  train frame {loss_f.item():.4f} chunk {loss_a.item():.4f}'
                f'   val frame {val_f:.4f} chunk {val_a:.4f}'
            )
    return {k: v.cpu() for k, v in net.state_dict().items()}

train_inv: not cached, will compute


In [19]:
inv3 = UWMxx(d3).to(device)
inv3.load_state_dict(train_inv(3))
inv3.eval();

    0  train frame 0.6606 chunk 0.9688   val frame 0.4854 chunk 1.1054
  500  train frame 0.0232 chunk 0.0097   val frame 0.0229 chunk 0.1575
 1000  train frame 0.0192 chunk 0.0119   val frame 0.0180 chunk 0.1520
 1500  train frame 0.0168 chunk 0.0037   val frame 0.0153 chunk 0.1466
 2000  train frame 0.0145 chunk 0.0054   val frame 0.0136 chunk 0.1483
 2500  train frame 0.0130 chunk 0.0046   val frame 0.0126 chunk 0.1441
 3000  train frame 0.0119 chunk 0.0028   val frame 0.0117 chunk 0.1450
 3500  train frame 0.0102 chunk 0.0040   val frame 0.0110 chunk 0.1490
 4000  train frame 0.0106 chunk 0.0033   val frame 0.0105 chunk 0.1466
 4500  train frame 0.0101 chunk 0.0027   val frame 0.0100 chunk 0.1440
 5000  train frame 0.0096 chunk 0.0024   val frame 0.0097 chunk 0.1435
 5500  train frame 0.0101 chunk 0.0025   val frame 0.0091 chunk 0.1426
 6000  train frame 0.0082 chunk 0.0015   val frame 0.0088 chunk 0.1450
 6500  train frame 0.0077 chunk 0.0019   val frame 0.0084 chunk 0.1461
 7000 

In [20]:
p, i = first_error(inv3, d3, False), first_error(inv3, d3, True)
print(
    f'3 + video + inverse rows   policy arm {p[:7].mean():.4f} grip {p[7]:.4f}   inverse arm {i[:7].mean():.4f} grip {i[7]:.4f}'
)
print('3 + video (step 4):        policy arm 0.0260 grip 0.0372   inverse arm 0.0259 grip 0.0372')

3 + video + inverse rows   policy arm 0.0268 grip 0.0391   inverse arm 0.0269 grip 0.0405
3 + video (step 4):        policy arm 0.0260 grip 0.0372   inverse arm 0.0259 grip 0.0372


### step 6 — break the shortcut: the inverse rows see **no** past frames

Step 5: still policy = inverse. On 3 demos the past frames say which demo and moment this is, so the chunk replays it and never needs t+16. Now the 32 inverse rows get all 4 past frames black (the "no past" marker): only joints + the true t+16 frame, a real inverse-dynamics job. At test the chain is two passes: imagine t+16 from the past, then act with the past hidden.


In [13]:
def batch_loss_idm(
    net: UWMxx, dn, rows: torch.Tensor, has_act: torch.Tensor, inv: torch.Tensor
) -> tuple:
    # batch_loss_inv; rows with inv True also get all 4 past frames black
    clean_f = cut(dn.TOP[dn.AFTER[rows, 0]].float() / 255 * 2 - 1)  # [B, 64, 768]
    clean_a = (dn.ACTION[dn.CHUNK[rows]] - dn.A_MEAN) / dn.A_STD  # [B, 16, 8]
    noise_f, noise_a = torch.randn_like(clean_f), torch.randn_like(clean_a)
    tau_v, tau_a = torch.rand(2, len(rows), device=device)
    tau_v = torch.where(inv, 1.0, tau_v)  # [B], 1 on the inverse-job rows
    tau_a = tau_a * has_act  # [B], 0 on video-only rows
    x_f = (1 - tau_v[:, None, None]) * noise_f + tau_v[:, None, None] * clean_f
    x_a = (1 - tau_a[:, None, None]) * noise_a + tau_a[:, None, None] * clean_a
    seen = (
        dn.TOP[dn.PAST[rows]].float() / 255 * ~inv[:, None, None, None, None]
    )  # inverse rows: black
    f_hat, a_hat = UWM.forward(net, seen, dn.STATE[rows], x_f, x_a, tau_v, tau_a)
    keep = (dn.REAL[rows] & has_act[:, None])[..., None].float()  # [B, 16, 1]
    loss_a = (((a_hat - clean_a) ** 2) * keep).sum() / (keep.sum() * 8)
    loss_f = ((f_hat - clean_f) ** 2).mean((1, 2))[~inv].mean()  # per row, given-frame rows out
    return loss_f, loss_a


torch.manual_seed(1)
print(
    'untrained frame {:.3f}  chunk {:.3f}'.format(
        *batch_loss_idm(UWMxx(d3).to(device), d3, rows, has_act, inv)
    )
)

untrained frame 0.654  chunk 0.981


In [14]:
@cache
def train_idm(n_lab: int, seed: int = 0, steps: int = 10000, lr: float = 1e-3) -> dict:
    # train_inv with batch_loss_idm
    dn, rows_lab = data_for(n_lab)
    rows_vid = torch.tensor(
        np.flatnonzero(~is_val & ~np.isin(episode, labelled[:n_lab])), device=device
    )
    i = torch.arange(128, device=device)
    has_act, inv = i < 64, (i >= 32) & (i < 64)
    torch.manual_seed(seed)
    net = UWMxx(dn).to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=lr)
    for step in range(steps + 1):
        rows = torch.cat([rows_lab[torch.randint(0, len(rows_lab), (64,), device=device)],
                          rows_vid[torch.randint(0, len(rows_vid), (64,), device=device)]])  # fmt: skip
        with torch.autocast('cuda', dtype=torch.bfloat16):
            loss_f, loss_a = batch_loss_idm(net, dn, rows, has_act, inv)
        opt.zero_grad()
        (loss_f + loss_a).backward()
        opt.step()
        if step % 500 == 0:
            val_f, val_a = val_loss_xx(net, dn, VAL)
            print(
                f'{step:5d}  train frame {loss_f.item():.4f} chunk {loss_a.item():.4f}'
                f'   val frame {val_f:.4f} chunk {val_a:.4f}'
            )
    return {k: v.cpu() for k, v in net.state_dict().items()}

train_idm: not cached, will compute


In [15]:
idm3 = UWMxx(d3).to(device)
idm3.load_state_dict(train_idm(3))
idm3.eval();

    0  train frame 0.6606 chunk 0.9688   val frame 0.4854 chunk 1.1054
  500  train frame 0.0227 chunk 0.0093   val frame 0.0226 chunk 0.1562
 1000  train frame 0.0193 chunk 0.0072   val frame 0.0179 chunk 0.1514
 1500  train frame 0.0167 chunk 0.0049   val frame 0.0152 chunk 0.1439
 2000  train frame 0.0144 chunk 0.0027   val frame 0.0136 chunk 0.1429
 2500  train frame 0.0129 chunk 0.0030   val frame 0.0124 chunk 0.1423
 3000  train frame 0.0118 chunk 0.0037   val frame 0.0115 chunk 0.1420
 3500  train frame 0.0102 chunk 0.0030   val frame 0.0109 chunk 0.1407
 4000  train frame 0.0106 chunk 0.0031   val frame 0.0107 chunk 0.1436
 4500  train frame 0.0102 chunk 0.0043   val frame 0.0100 chunk 0.1426
 5000  train frame 0.0095 chunk 0.0017   val frame 0.0096 chunk 0.1402
 5500  train frame 0.0102 chunk 0.0017   val frame 0.0092 chunk 0.1423
 6000  train frame 0.0082 chunk 0.0016   val frame 0.0089 chunk 0.1428
 6500  train frame 0.0077 chunk 0.0020   val frame 0.0085 chunk 0.1466
 7000 

In [16]:
@torch.no_grad()
def first_error_idm(net: UWMxx, dn, steps: int = 10) -> torch.Tensor:
    # first_error's inverse job with the past frames black: joints + true t+16 only; out: [8]
    torch.manual_seed(0)
    errs = []
    for j in range(0, len(VAL), 256):
        rows = VAL[j : j + 256]
        B = len(rows)
        x_f = cut(dn.TOP[dn.AFTER[rows, 0]].float() / 255 * 2 - 1)  # where it must end up
        tau_v = torch.ones(B, device=device)
        x_a = torch.randn(B, 16, 8, device=device)
        seen = torch.zeros(B, 4, 3, 128, 128, device=device)  # no past
        for k in range(steps):
            tau_a = torch.full((B,), k / steps, device=device)
            with torch.autocast('cuda', dtype=torch.bfloat16):
                _, v_a = net(seen, dn.STATE[rows], x_f, x_a, tau_v, tau_a)
            x_a = x_a + v_a.float() / steps
        errs.append(((x_a * dn.A_STD + dn.A_MEAN)[:, 0] - dn.ACTION[rows]).abs())
    return torch.cat(errs).mean(0)

In [17]:
p, i = first_error(idm3, d3, False), first_error_idm(idm3, d3)
print(f'policy job (past given, frame noise)   arm {p[:7].mean():.4f} grip {p[7]:.4f}')
print(f'IDM job (past black, true t+16)        arm {i[:7].mean():.4f} grip {i[7]:.4f}')
print('step 5 policy: arm 0.0268 grip 0.0391;  10 labelled policy: arm 0.0114 grip 0.0353')

policy job (past given, frame noise)   arm 0.0257 grip 0.0313
IDM job (past black, true t+16)        arm 0.0256 grip 0.0402
step 5 policy: arm 0.0268 grip 0.0391;  10 labelled policy: arm 0.0114 grip 0.0353


The first action sits next to the current joints, so joints alone nearly fix it. t+16 should matter most at the **end** of the chunk: arm error at every chunk step, both jobs.

In [14]:
@torch.no_grad()
def chunk_error(net: UWMxx, dn, idm: bool, steps: int = 10) -> torch.Tensor:
    # out: [16] val arm |error| at each chunk step, radians, only where the chunk is inside the demo
    # idm False: policy job (past given, frame noise); True: past black, true t+16 at tau_v 1
    torch.manual_seed(0)
    num, den = torch.zeros(16, device=device), torch.zeros(16, device=device)
    for j in range(0, len(VAL), 256):
        rows = VAL[j : j + 256]
        B = len(rows)
        seen = dn.TOP[dn.PAST[rows]].float() / 255 * (not idm)
        if idm:
            x_f = cut(dn.TOP[dn.AFTER[rows, 0]].float() / 255 * 2 - 1)
        else:
            x_f = torch.randn(B, 64, 768, device=device)
        tau_v = torch.full((B,), float(idm), device=device)
        x_a = torch.randn(B, 16, 8, device=device)
        for k in range(steps):
            tau_a = torch.full((B,), k / steps, device=device)
            with torch.autocast('cuda', dtype=torch.bfloat16):
                _, v_a = net(seen, dn.STATE[rows], x_f, x_a, tau_v, tau_a)
            x_a = x_a + v_a.float() / steps
        err = ((x_a * dn.A_STD + dn.A_MEAN) - dn.ACTION[dn.CHUNK[rows]])[..., :7].abs().mean(-1)
        num += (err * dn.REAL[rows]).sum(0)
        den += dn.REAL[rows].sum(0)
    return num / den

In [19]:
pol, idm = chunk_error(idm3, d3, False), chunk_error(idm3, d3, True)
for k in (0, 3, 7, 11, 15):
    print(f'chunk step {k:2d}   policy {pol[k]:.4f}   IDM {idm[k]:.4f}')

chunk step  0   policy 0.0257   IDM 0.0256
chunk step  3   policy 0.0269   IDM 0.0265
chunk step  7   policy 0.0281   IDM 0.0279
chunk step 11   policy 0.0290   IDM 0.0287
chunk step 15   policy 0.0301   IDM 0.0299


In [20]:
@torch.no_grad()
def chunk_error_blind(net: UWMxx, dn, steps: int = 10) -> torch.Tensor:
    # chunk_error with no picture at all: past black AND frame left as noise; joints only
    torch.manual_seed(0)
    num, den = torch.zeros(16, device=device), torch.zeros(16, device=device)
    for j in range(0, len(VAL), 256):
        rows = VAL[j : j + 256]
        B = len(rows)
        seen = torch.zeros(B, 4, 3, 128, 128, device=device)
        x_f, tau_v = torch.randn(B, 64, 768, device=device), torch.zeros(B, device=device)
        x_a = torch.randn(B, 16, 8, device=device)
        for k in range(steps):
            tau_a = torch.full((B,), k / steps, device=device)
            with torch.autocast('cuda', dtype=torch.bfloat16):
                _, v_a = net(seen, dn.STATE[rows], x_f, x_a, tau_v, tau_a)
            x_a = x_a + v_a.float() / steps
        err = ((x_a * dn.A_STD + dn.A_MEAN) - dn.ACTION[dn.CHUNK[rows]])[..., :7].abs().mean(-1)
        num += (err * dn.REAL[rows]).sum(0)
        den += dn.REAL[rows].sum(0)
    return num / den


blind = chunk_error_blind(idm3, d3)
for k in (0, 7, 15):
    print(
        f'chunk step {k:2d}   joints only {blind[k]:.4f}   (policy {pol[k]:.4f}, IDM {idm[k]:.4f})'
    )

chunk step  0   joints only 0.0262   (policy 0.0257, IDM 0.0256)
chunk step  7   joints only 0.0284   (policy 0.0281, IDM 0.0279)
chunk step 15   joints only 0.0303   (policy 0.0301, IDM 0.0299)


In [25]:
for name, (net, dn) in {'3 alone': (few3, d3), '10 alone': (lab, d)}.items():
    b, p = chunk_error_blind(net, dn), chunk_error(net, dn, False)
    print(
        f'{name:9s} step 0 / 15   joints only {b[0]:.4f} / {b[15]:.4f}   policy {p[0]:.4f} / {p[15]:.4f}'
    )

3 alone   step 0 / 15   joints only 0.0281 / 0.0340   policy 0.0273 / 0.0332
10 alone  step 0 / 15   joints only 0.0126 / 0.0177   policy 0.0114 / 0.0160


### step 7 — no joints: the action half has to read the pictures

Step 6: on 3 demos the chunk reads only the joints (no picture at all: same error). The joints say where in which demo the arm is, so the chunk replays it. Take them away everywhere, training and sim: the joints layer only ever sees zeros, so its token is one constant ("no joints"). Then the arm's pose and the cube both have to come from pixels, the part the 87 video demos train. Two arms on 3 labelled demos: alone (step 2's batches) and + video (step 3's 64 + 64).

In [16]:
from torch import nn


class NoJoints(nn.Module):
    # stands in for net.joints_in: the same Linear, always fed zeros (normalised joints = 0)
    def __init__(self, lin: nn.Linear):
        super().__init__()
        self.lin = lin

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.lin(torch.zeros_like(x))


def uwm_nj(dn) -> UWMxx:
    # a UWMxx that never sees the joints
    net = UWMxx(dn).to(device)
    net.joints_in = NoJoints(net.joints_in)
    return net

In [17]:
@cache
def train_nj(n_lab: int, video: bool, seed: int = 0, steps: int = 10000, lr: float = 1e-3) -> dict:
    # train_few (video False: 64 labelled rows) or train_mix (True: + 64 video-only), no joints
    dn, rows_lab = data_for(n_lab)
    rows_vid = torch.tensor(
        np.flatnonzero(~is_val & ~np.isin(episode, labelled[:n_lab])), device=device
    )
    has_act = torch.arange(128 if video else 64, device=device) < 64
    torch.manual_seed(seed)
    net = uwm_nj(dn)
    opt = torch.optim.AdamW(net.parameters(), lr=lr)
    for step in range(steps + 1):
        rows = rows_lab[torch.randint(0, len(rows_lab), (64,), device=device)]
        if video:
            rows = torch.cat(
                [rows, rows_vid[torch.randint(0, len(rows_vid), (64,), device=device)]]
            )
        with torch.autocast('cuda', dtype=torch.bfloat16):
            loss_f, loss_a = batch_loss_mix(net, dn, rows, has_act)
        opt.zero_grad()
        (loss_f + loss_a).backward()
        opt.step()
        if step % 500 == 0:
            val_f, val_a = val_loss_xx(net, dn, VAL)
            print(
                f'{step:5d}  train frame {loss_f.item():.4f} chunk {loss_a.item():.4f}'
                f'   val frame {val_f:.4f} chunk {val_a:.4f}'
            )
    return {k: v.cpu() for k, v in net.state_dict().items()}

train_nj: cached, 2 call(s) stored


In [18]:
nj3 = uwm_nj(d3)
nj3.load_state_dict(train_nj(3, video=False))
nj3.eval();

    0  train frame 0.6581 chunk 0.9688   val frame 0.4854 chunk 1.1046
  500  train frame 0.0255 chunk 0.0825   val frame 0.0265 chunk 0.2489
 1000  train frame 0.0183 chunk 0.0387   val frame 0.0218 chunk 0.2111
 1500  train frame 0.0152 chunk 0.0059   val frame 0.0189 chunk 0.2948
 2000  train frame 0.0118 chunk 0.0053   val frame 0.0175 chunk 0.3254
 2500  train frame 0.0130 chunk 0.0017   val frame 0.0164 chunk 0.3292
 3000  train frame 0.0095 chunk 0.0027   val frame 0.0163 chunk 0.3492
 3500  train frame 0.0085 chunk 0.0013   val frame 0.0165 chunk 0.3659
 4000  train frame 0.0083 chunk 0.0022   val frame 0.0169 chunk 0.3836
 4500  train frame 0.0071 chunk 0.0008   val frame 0.0169 chunk 0.3750
 5000  train frame 0.0065 chunk 0.0013   val frame 0.0174 chunk 0.3945
 5500  train frame 0.0060 chunk 0.0005   val frame 0.0178 chunk 0.3811
 6000  train frame 0.0055 chunk 0.0009   val frame 0.0183 chunk 0.4013
 6500  train frame 0.0055 chunk 0.0019   val frame 0.0189 chunk 0.4056
 7000 

In [19]:
nj3v = uwm_nj(d3)
nj3v.load_state_dict(train_nj(3, video=True))
nj3v.eval();

    0  train frame 0.6596 chunk 0.9688   val frame 0.4854 chunk 1.1053
  500  train frame 0.0260 chunk 0.0624   val frame 0.0277 chunk 0.2505
 1000  train frame 0.0242 chunk 0.0364   val frame 0.0221 chunk 0.2539
 1500  train frame 0.0193 chunk 0.0090   val frame 0.0182 chunk 0.2396
 2000  train frame 0.0167 chunk 0.0033   val frame 0.0160 chunk 0.2348
 2500  train frame 0.0140 chunk 0.0058   val frame 0.0144 chunk 0.2361
 3000  train frame 0.0127 chunk 0.0019   val frame 0.0127 chunk 0.2467
 3500  train frame 0.0105 chunk 0.0016   val frame 0.0116 chunk 0.2509
 4000  train frame 0.0106 chunk 0.0017   val frame 0.0110 chunk 0.2623
 4500  train frame 0.0095 chunk 0.0015   val frame 0.0102 chunk 0.2598
 5000  train frame 0.0094 chunk 0.0008   val frame 0.0097 chunk 0.2591
 5500  train frame 0.0089 chunk 0.0025   val frame 0.0091 chunk 0.2674
 6000  train frame 0.0077 chunk 0.0007   val frame 0.0088 chunk 0.2623
 6500  train frame 0.0076 chunk 0.0047   val frame 0.0092 chunk 0.2596
 7000 

In [20]:
pics = {'no joints, 3 alone': nj3, 'no joints, 3 + video': nj3v}
for name, net in pics.items():
    b, p = chunk_error_blind(net, d3), chunk_error(net, d3, False)
    print(
        f'{name:21s} step 0 / 15   no picture {b[0]:.4f} / {b[15]:.4f}   pictures {p[0]:.4f} / {p[15]:.4f}'
    )
print('with joints, 3 alone: pictures 0.0273 / 0.0332;  10 alone: 0.0114 / 0.0160')

no joints, 3 alone    step 0 / 15   no picture 0.1328 / 0.1332   pictures 0.0449 / 0.0542
no joints, 3 + video  step 0 / 15   no picture 0.1148 / 0.1203   pictures 0.0395 / 0.0404
with joints, 3 alone: pictures 0.0273 / 0.0332;  10 alone: 0.0114 / 0.0160


In [21]:
@cache
def sweep_nj(
    n_exec: int, n_lab: int, video: bool, train_seed: int = 0, train_steps: int = 10000
) -> dict:
    # sweep_few with the weights train_nj gives (rollout's obs.state never reaches the model)
    dn, _ = data_for(n_lab)
    net = uwm_nj(dn)
    with contextlib.redirect_stdout(io.StringIO()):
        net.load_state_dict(train_nj(n_lab, video, seed=train_seed, steps=train_steps))
    net.eval()
    torch.manual_seed(n_exec)  # the sampling noise, fixed per call
    res = {seed: rollout(env, net, dn, seed, n_exec) for seed in range(1000, 1020)}
    fails = Counter(mode for ok, mode, _ in res.values() if not ok)
    print(
        f'video {video!s:5s} n_exec {n_exec:2d}: {sum(ok for ok, _, _ in res.values()):2d}/20   failures {dict(fails)}'
    )
    return res

sweep_nj: not cached, will compute


In [22]:
sweeps_nj = {
    (video, n_exec): sweep_nj(n_exec, 3, video) for video in [False, True] for n_exec in [8, 16]
}
print('with joints: 3 alone 6, 6;  3 + video 8, 7;  10 alone 16, 14')

video False n_exec  8: 10/20   failures {'no_grasp': 10}
video False n_exec 16:  9/20   failures {'no_grasp': 11}
video True  n_exec  8:  5/20   failures {'no_grasp': 14, 'wrong_target': 1}
video True  n_exec 16:  8/20   failures {'no_grasp': 11, 'dropped': 1}
with joints: 3 alone 6, 6;  3 + video 8, 7;  10 alone 16, 14


### step 8 — the +video run spiked at its very last step; retrain it with cosine decay + grad clipping

Step 7's +video log was smooth to 9500 (val frame 0.0073), then step 10000 jumped (0.0175): the saved weights are mid-spike. Same run, with the lr following a cosine from 1e-3 to 0 and gradients clipped at norm 1, so training ends at a tiny lr. Only the +video arm for now (the alone arm didn't spike).

In [18]:
@cache
def train_nj_cos(
    n_lab: int, video: bool, seed: int = 0, steps: int = 10000, lr: float = 1e-3
) -> dict:
    # train_nj with a cosine lr (lr -> 0 at the last step) and grad-norm clipping at 1
    dn, rows_lab = data_for(n_lab)
    rows_vid = torch.tensor(
        np.flatnonzero(~is_val & ~np.isin(episode, labelled[:n_lab])), device=device
    )
    has_act = torch.arange(128 if video else 64, device=device) < 64
    torch.manual_seed(seed)
    net = uwm_nj(dn)
    opt = torch.optim.AdamW(net.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
    for step in range(steps + 1):
        rows = rows_lab[torch.randint(0, len(rows_lab), (64,), device=device)]
        if video:
            rows = torch.cat(
                [rows, rows_vid[torch.randint(0, len(rows_vid), (64,), device=device)]]
            )
        with torch.autocast('cuda', dtype=torch.bfloat16):
            loss_f, loss_a = batch_loss_mix(net, dn, rows, has_act)
        opt.zero_grad()
        (loss_f + loss_a).backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
        sched.step()
        if step % 500 == 0:
            val_f, val_a = val_loss_xx(net, dn, VAL)
            print(
                f'{step:5d}  lr {sched.get_last_lr()[0]:.1e}  train frame {loss_f.item():.4f} chunk {loss_a.item():.4f}'
                f'   val frame {val_f:.4f} chunk {val_a:.4f}'
            )
    return {k: v.cpu() for k, v in net.state_dict().items()}

train_nj_cos: not cached, will compute


In [19]:
nj3v_cos = uwm_nj(d3)
nj3v_cos.load_state_dict(train_nj_cos(3, video=True))
nj3v_cos.eval();

    0  lr 1.0e-03  train frame 0.6596 chunk 0.9688   val frame 0.4854 chunk 1.1053
  500  lr 9.9e-04  train frame 0.0253 chunk 0.0539   val frame 0.0269 chunk 0.2638
 1000  lr 9.8e-04  train frame 0.0238 chunk 0.0490   val frame 0.0218 chunk 0.2531
 1500  lr 9.5e-04  train frame 0.0194 chunk 0.0146   val frame 0.0184 chunk 0.2858
 2000  lr 9.0e-04  train frame 0.0174 chunk 0.0035   val frame 0.0165 chunk 0.2752
 2500  lr 8.5e-04  train frame 0.0140 chunk 0.0020   val frame 0.0146 chunk 0.2654
 3000  lr 7.9e-04  train frame 0.0134 chunk 0.0014   val frame 0.0132 chunk 0.2734
 3500  lr 7.3e-04  train frame 0.0111 chunk 0.0011   val frame 0.0122 chunk 0.2725
 4000  lr 6.5e-04  train frame 0.0109 chunk 0.0014   val frame 0.0114 chunk 0.2804
 4500  lr 5.8e-04  train frame 0.0099 chunk 0.0004   val frame 0.0106 chunk 0.2789
 5000  lr 5.0e-04  train frame 0.0097 chunk 0.0008   val frame 0.0101 chunk 0.2781
 5500  lr 4.2e-04  train frame 0.0094 chunk 0.0006   val frame 0.0096 chunk 0.2790
 600

In [20]:
b, p = chunk_error_blind(nj3v_cos, d3), chunk_error(nj3v_cos, d3, False)
print(
    f'no joints, 3 + video, cosine   step 0 / 15   no picture {b[0]:.4f} / {b[15]:.4f}   pictures {p[0]:.4f} / {p[15]:.4f}'
)
print('step 7: alone pictures 0.0449 / 0.0542;  + video (spiked) 0.0395 / 0.0404')

no joints, 3 + video, cosine   step 0 / 15   no picture 0.1842 / 0.1861   pictures 0.0345 / 0.0396
step 7: alone pictures 0.0449 / 0.0542;  + video (spiked) 0.0395 / 0.0404


In [21]:
@cache
def sweep_nj_cos(
    n_exec: int, n_lab: int, video: bool, train_seed: int = 0, train_steps: int = 10000
) -> dict:
    # sweep_nj with the weights train_nj_cos gives
    dn, _ = data_for(n_lab)
    net = uwm_nj(dn)
    with contextlib.redirect_stdout(io.StringIO()):
        net.load_state_dict(train_nj_cos(n_lab, video, seed=train_seed, steps=train_steps))
    net.eval()
    torch.manual_seed(n_exec)  # the sampling noise, fixed per call
    res = {seed: rollout(env, net, dn, seed, n_exec) for seed in range(1000, 1020)}
    fails = Counter(mode for ok, mode, _ in res.values() if not ok)
    print(
        f'video {video!s:5s} n_exec {n_exec:2d}: {sum(ok for ok, _, _ in res.values()):2d}/20   failures {dict(fails)}'
    )
    return res

sweep_nj_cos: not cached, will compute


In [22]:
sweeps_nj_cos = {n_exec: sweep_nj_cos(n_exec, 3, video=True) for n_exec in [8, 16]}
print('step 7, no joints: alone 10, 9;  + video (spiked) 5, 8')

video True  n_exec  8:  8/20   failures {'no_grasp': 11, 'dropped': 1}
video True  n_exec 16:  9/20   failures {'no_grasp': 10, 'dropped': 1}
step 7, no joints: alone 10, 9;  + video (spiked) 5, 8
